# Grounded VQG reproduction -- full training on Colab Pro

Reproduces *Automatic Generation of Grounded Visual Questions* (Zhang et al., IJCAI 2017), architecture-faithful, with two library substitutions for deprecated dependencies:
- Original Torch/Lua DenseCap -> [`soloist97/densecap-pytorch`](https://github.com/soloist97/densecap-pytorch) (only affects the diversity/quality of the *input* captions, not the VQG model architecture -- see this project's README for the full writeup).
- Original NeuralTalk2 baseline -> not reproduced here (this notebook trains the paper's actual model only).

**Before running:** Runtime -> Change runtime type -> GPU (A100 recommended if your Colab Pro quota allows; T4 works, just slower for the DenseCap pass).

**This will take hours.** COCO images alone are ~26GB, and DenseCap inference + full training over VQA v1 (764k questions) or Visual7W (327k QA pairs) is not a quick job. Everything below writes intermediate artifacts to Google Drive so you can stop and resume.

## 0. Config -- fill these in

In [ ]:
GITHUB_REPO_URL = "https://github.com/malimustafaa/Automatic-Generation-of-Grounded-Visual-Questions.git"
DATASET = "vqa"  # "vqa" or "visual7w"
DRIVE_ROOT = "/content/drive/MyDrive/grounded-vqg-reproduction"  # all downloads/checkpoints persist here

# DenseCap-pytorch pretrained checkpoint -- the authors distribute it via OneDrive/BaiduYun
# (see https://github.com/soloist97/densecap-pytorch#pretrained-model), not a direct URL we
# can curl. Download it once by hand, drop it in Drive, and point this at it:
DENSECAP_CHECKPOINT = f"{DRIVE_ROOT}/densecap_pytorch/checkpoint.pth.tar"
DENSECAP_CONFIG = f"{DRIVE_ROOT}/densecap_pytorch/config.json"  # ships alongside the checkpoint in the repo's release

## 1. Mount Drive + clone this repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)

!git clone {GITHUB_REPO_URL} /content/grounded-vqg-reproduction
%cd /content/grounded-vqg-reproduction

## 2. Install dependencies

In [ ]:
!pip install -q pyyaml pycocoevalcap tqdm
# torch/torchvision/numpy/pillow are already present on Colab images.

# Run the smoke test first -- if this fails, nothing downstream will work either,
# and it takes seconds rather than hours to find out.
!python -m tests.smoke_test

## 3. Download GloVe + COCO images (persisted to Drive)

In [ ]:
!bash scripts/download_glove.sh {DRIVE_ROOT}/data
!bash scripts/download_coco_images.sh {DRIVE_ROOT}/data/coco

## 4. Download & flatten the question dataset (VQA v1 or Visual7W)

In [ ]:
if DATASET == "vqa":
    !python scripts/prepare_vqa.py --dest_dir {DRIVE_ROOT}/data/vqa_raw --out {DRIVE_ROOT}/data/questions.json
else:
    !python scripts/prepare_visual7w.py --dest_dir {DRIVE_ROOT}/data/visual7w_raw --coco_dir {DRIVE_ROOT}/data/coco --out {DRIVE_ROOT}/data/questions.json

## 5. Extract frozen VGG-16 image features (300-d, paper Sec 3.2/4.4)

In [ ]:
!python scripts/extract_image_features.py \
  --questions {DRIVE_ROOT}/data/questions.json \
  --image_root {DRIVE_ROOT}/data/coco \
  --out_dir {DRIVE_ROOT}/data/image_features

## 6. DenseCap candidate captions (the one library substitution -- see README)

Clones `soloist97/densecap-pytorch` and runs its `describe.py` over every image, then reshapes the output into this project's candidate-caption format. Requires `DENSECAP_CHECKPOINT`/`DENSECAP_CONFIG` from cell 0 to already exist in Drive (download once by hand from the densecap-pytorch README's OneDrive/BaiduYun link -- this can't be automated with a plain URL).

In [ ]:
!git clone https://github.com/soloist97/densecap-pytorch.git /content/densecap-pytorch
assert os.path.exists(DENSECAP_CHECKPOINT), (
    "Missing DenseCap checkpoint -- download it by hand from "
    "https://github.com/soloist97/densecap-pytorch#pretrained-model and place it at "
    f"{DENSECAP_CHECKPOINT} (and its config.json alongside it) before re-running this cell."
)

!python scripts/run_densecap.py \
  --densecap_repo /content/densecap-pytorch \
  --config_json {DENSECAP_CONFIG} \
  --checkpoint {DENSECAP_CHECKPOINT} \
  --img_dir {DRIVE_ROOT}/data/coco/train2014 \
  --result_dir {DRIVE_ROOT}/data/densecap_raw \
  --questions {DRIVE_ROOT}/data/questions.json \
  --out {DRIVE_ROOT}/data/densecap_candidates.json

## 7. Build the final training manifest

In [ ]:
!python scripts/build_manifest.py \
  --questions {DRIVE_ROOT}/data/questions.json \
  --features_dir {DRIVE_ROOT}/data/image_features \
  --candidates {DRIVE_ROOT}/data/densecap_candidates.json \
  --out {DRIVE_ROOT}/data/manifest.json

## 8. Train

Paper-given: batch size 64, 128 epochs (VQA) / 64 epochs (Visual7W) -- `configs/default.yaml` defaults to 128; pass `--epochs 64` for Visual7W. All hyperparameters the paper never specifies (learning rate, hidden sizes, etc.) live in that same config file with inline comments -- edit there, not here, if you want to sweep them.

In [ ]:
epochs_arg = "" if DATASET == "vqa" else "--epochs 64"
!python -m src.train \
  --config configs/default.yaml \
  --manifest {DRIVE_ROOT}/data/manifest.json \
  --glove {DRIVE_ROOT}/data/glove.840B.300d.txt \
  --out_dir {DRIVE_ROOT}/checkpoints \
  {epochs_arg}

## 9. Generate + evaluate

Reproduces the paper's Fig. 3 precision/recall sweep over N=1..6 generated questions per image (`eval/evaluate.py::sweep_num_questions`). Fill in a checkpoint path and a held-out slice of the manifest before running.

In [ ]:
import json, torch
from src.vocab import Vocab
from src.embeddings import build_embedding_matrix
from src.model import GroundedVQGModel
from src.bigram_lm import KneserNeyBigram
from src.dataset import tokenize
from src.generate import generate_questions
from eval.evaluate import sweep_num_questions, group_references_by_image

CKPT_PATH = f"{DRIVE_ROOT}/checkpoints/checkpoint_epoch128.pt"  # adjust to the epoch you want to evaluate
ckpt = torch.load(CKPT_PATH, map_location="cpu")
vocab = Vocab(); vocab.idx2word = ckpt["vocab"]; vocab.word2idx = {w: i for i, w in enumerate(vocab.idx2word)}

embedding = build_embedding_matrix(vocab, dim=300)  # shapes only; real weights load via state_dict below
model = GroundedVQGModel(embedding, vocab_size=len(vocab),
                          type_hidden=ckpt["cfg"]["type_selector_hidden"],
                          decoder_hidden=ckpt["cfg"]["decoder_hidden"])
model.load_state_dict(ckpt["model"])
model.eval()

with open(f"{DRIVE_ROOT}/data/manifest.json") as f:
    manifest = json.load(f)
eval_records = manifest[:500]  # small held-out slice for a first pass; widen once this runs cleanly

bigram_lm = KneserNeyBigram(discount=0.75).fit([tokenize(r["question"]) for r in manifest])

generated_pool, references = {}, {}
for r in eval_records:
    image_id = str(r["image_id"])
    import numpy as np
    feat = torch.from_numpy(np.load(r["image_feat_path"])).float()
    qs = generate_questions(model, vocab, bigram_lm, feat, r["candidates"],
                             num_questions=6, beta=ckpt["cfg"]["bigram_beta"])
    generated_pool.setdefault(image_id, []).extend(qs)
    references.setdefault(image_id, []).append(r["question"])

results = sweep_num_questions(references, generated_pool, max_n=6)
for n, scores in results.items():
    print(n, scores)